# Notebook 01 — Setup & Document Chunking

**Project:** Vietnamese University QA System (Topic 1)  
**Domain:** TDTU University Regulations  
**Steps:**
1. Mount Google Drive & install dependencies
2. Create output directory structure
3. Load 20 Vietnamese text files từ `data/text/`
4. Chunk **toàn bộ** tài liệu với Hybrid Agentic Chunking
5. Lưu tất cả chunks → `data/chunks/all_chunks.jsonl`
6. In thống kê

## 1. Mount Google Drive

In [ ]:
from pathlib import Path
import os

# Tự động tìm đường dẫn
if os.path.exists('/content/'):
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/NLP_Final/Source/')
else:
    BASE = "../"

print(f'Base directory: {BASE}')
print(os.listdir(BASE))

Mounted at /content/drive
Base directory: /content/drive/MyDrive/NLP_Final/Source
['.gitignore', '.env', 'requirements.txt', 'scripts', '.git', 'data', 'notebooks', 'models', 'results']


## 2. Install Dependencies

In [ ]:
%%capture
!pip install -q \
    transformers==4.45.0 \
    peft==0.13.0 \
    trl==0.11.4 \
    bitsandbytes==0.44.1 \
    accelerate==1.0.1 \
    datasets==3.1.0 \
    'huggingface-hub>=0.26.0' \
    sentence-transformers==3.2.1 \
    faiss-cpu==1.8.0 \
    langchain==0.3.7 \
    langchain-community==0.3.7 \
    sacrebleu==2.4.3 \
    rouge-score==0.1.2 \
    bert-score==0.3.13 \
    gradio==5.5.0 \
    'google-genai>=0.3.0' \
    python-dotenv \
    tqdm \
    google-genai \
    openai
print('Dependencies installed.')

## 3. Create Directory Structure

In [ ]:
import os

DIRS = [
    'data/text',
    'data/chunks',
    'data/qa_raw',
    'data/qa_filtered',
    'data/vector_store/faiss_index',
    'models/sft_checkpoint',
    'models/reward_model',
    'models/ppo_checkpoint',
    'results',
]

for d in DIRS:
    os.makedirs(f'{BASE}/{d}', exist_ok=True)

print('Directory structure ready:')
for d in DIRS:
    print(f'  {BASE}/{d}')

Directory structure ready:
  /content/drive/MyDrive/NLP_Final/Source/data/text
  /content/drive/MyDrive/NLP_Final/Source/data/chunks
  /content/drive/MyDrive/NLP_Final/Source/data/qa_raw
  /content/drive/MyDrive/NLP_Final/Source/data/qa_filtered
  /content/drive/MyDrive/NLP_Final/Source/data/vector_store/faiss_index
  /content/drive/MyDrive/NLP_Final/Source/models/sft_checkpoint
  /content/drive/MyDrive/NLP_Final/Source/models/reward_model
  /content/drive/MyDrive/NLP_Final/Source/models/ppo_checkpoint
  /content/drive/MyDrive/NLP_Final/Source/results


## 4. Upload Text Files to Drive

**Option A — Direct upload từ máy local:**  
Chạy cell dưới để upload.

**Option B — Đã có sẵn trên Drive:**  
Bỏ qua cell này nếu file `.txt` đã nằm trong `NLP_Final/Source/data/text/`.

In [ ]:
# Option A: Upload từ máy local
# from google.colab import files
# uploaded = files.upload()
# import shutil
# for fname in uploaded:
#     shutil.move(fname, f"{BASE}/data/text/{fname}")

# Kiểm tra file đã có
TEXT_DIR = f'{BASE}/data/text'
txt_files = sorted([f for f in os.listdir(TEXT_DIR) if f.endswith('.txt')])
print(f'Found {len(txt_files)} text files:')
for f in txt_files:
    size = os.path.getsize(f'{TEXT_DIR}/{f}')
    print(f'  {f}  ({size:,} bytes)')

Found 20 text files:
  (TV)-Hướng dẫn giao tiếp, ứng xử cho người học.txt  (13,942 bytes)
  11.2457-QD cap chung nhan ky su cu nhan uu tu.txt  (7,022 bytes)
  15.2018 - Quy dinh ve hoat dong tap su nghe nghiep.txt  (7,276 bytes)
  17.Thong tin phong ban.txt  (12,969 bytes)
  2021_Quy chế tổ chức và quản lý đào tạo.txt  (83,273 bytes)
  25.2020-Noi quy phong thi.txt  (14,732 bytes)
  Bộ Quy tắc ứng xử của người học.txt  (17,474 bytes)
  Hướng dẫn học bổng, khen thưởng, hỗ trợ người học và hỗ trợ khác.txt  (109,860 bytes)
  MÔ TẢ 5 ĐẶC ĐIỂM NHẬN DIỆN SINH VIÊN TDTU.txt  (3,390 bytes)
  Nhiệm vụ thực hiện 3 nội dung đạo đức.txt  (1,747 bytes)
  Nội dung và thang điểm đánh giá rèn luyện trình độ đại học.txt  (16,041 bytes)
  PHỤ LỤC NHỮNG ĐIỀU SINH VIÊN KHÔNG ĐƯỢC LÀM; NỘI DUNG VI PHẠM VÀ HÌNH THỨC XỬ LÝ SINH VIÊN.txt  (18,412 bytes)
  QD 2793 - Nội quy thi trực 

## 5. Hybrid Agentic Chunking

**Pipeline theo đúng quy trình agentic chunking:**

| Bước | Phương pháp | Mục đích |
|---|---|---|
| 1 | `clean_text()` | Chuẩn hóa văn bản thô |
| 2 | `split_text() + merge_short_chunks() + split_large_chunk()` | Tách theo ranh giới đoạn văn \n\n, , gộp các chunk quá ngắn (<80 chars), Tách chunk dài (>800 chars) bằng phương pháp mechanical |
| 3 | LLM **metadata labeling** | Sinh `title` + `summary` cho mỗi chunk |
| 4 | Tạo `embed_text = title + summary + text` | Text giàu ngữ nghĩa để embedding|

**Hỗ trợ 2 LLM provider:**

| Provider | Model | Rate limit free | Chi phí |
|---|---|---|---|
| **Gemini** | `gemini-2.0-flash` | 15 RPM | Miễn phí |
| **DeepSeek** | `deepseek-chat` | Cao hơn | ~$0.28/1M tokens |

In [ ]:
import os, sys
from dotenv import load_dotenv

# Reload chunking module với code mới nhất (quan trọng nếu đã sửa chunking.py)
if 'scripts.chunking' in sys.modules:
    import importlib, scripts.chunking
    importlib.reload(scripts.chunking)

sys.path.insert(0, os.path.abspath(BASE))
from scripts.chunking import HybridChunker

load_dotenv(os.path.join(BASE, '.env'))


# CHỌN PROVIDER: 'gemini' hoặc 'deepseek'
PROVIDER = 'deepseek'

if PROVIDER == 'gemini':
    from google import genai
    GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY', 'YOUR_GEMINI_KEY')
    client   = genai.Client(api_key=GEMINI_API_KEY)
    model_id = 'gemini-2.0-flash'
    print(f'Provider: Gemini | Model: {model_id}')

elif PROVIDER == 'deepseek':
    try:
        from openai import OpenAI
    except ImportError:
        import subprocess, sys as _sys
        subprocess.check_call([_sys.executable, '-m', 'pip', 'install', 'openai', '-q'])
        from openai import OpenAI
    DEEPSEEK_API_KEY = os.environ.get('DEEPSEEK_API_KEY', 'YOUR_DEEPSEEK_KEY')
    client   = OpenAI(api_key=DEEPSEEK_API_KEY, base_url='https://api.deepseek.com')
    model_id = 'deepseek-chat'
    print(f'Provider: DeepSeek | Model: {model_id}')

else:
    raise ValueError(f"PROVIDER phải là 'gemini' hoặc 'deepseek', nhận: '{PROVIDER}'")

chunker = HybridChunker(client, model_id=model_id, provider=PROVIDER)
print('HybridChunker ready.')

Provider: DeepSeek | Model: deepseek-chat
HybridChunker ready.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(BASE))

from scripts.chunking import clean_text, split_text, merge_short_chunks, split_large_chunk, CHUNK_MIN, CHUNK_MAX

files = [
    BASE / Path("data/text/Hướng dẫn học bổng, khen thưởng, hỗ trợ người học và hỗ trợ khác.txt"),
    BASE / Path("data/text/(TV)-Hướng dẫn giao tiếp, ứng xử cho người học.txt"),
]

for fpath in files:
    with open(fpath, encoding="utf-8") as f:
        text = clean_text(f.read())

    raw    = split_text(text)
    merged = merge_short_chunks(raw)
    final  = [sub for c in merged for sub in split_large_chunk(c)]

    lengths = [len(c) for c in final]
    print(f"\n{os.path.basename(fpath)}")
    print(f"  split  : {len(raw)} chunks")
    print(f"  after merge+split: {len(final)} chunks")
    print(f"  too_small (<{CHUNK_MIN}): {sum(1 for l in lengths if l < CHUNK_MIN)}")
    print(f"  good ({CHUNK_MIN}-{CHUNK_MAX})  : {sum(1 for l in lengths if CHUNK_MIN <= l <= CHUNK_MAX)}")
    print(f"  too_large (>{CHUNK_MAX}): {sum(1 for l in lengths if l > CHUNK_MAX)}")
    print(f"  avg: {sum(lengths)//len(lengths)} | min: {min(lengths)} | max: {max(lengths)}")


  Chunking: 36 chunks (by paragraph breaks)

Hướng dẫn học bổng, khen thưởng, hỗ trợ người học và hỗ trợ khác.txt
  split  : 36 chunks
  after merge+split: 168 chunks
  too_small (<80): 12
  good (80-800)  : 122
  too_large (>800): 34
  avg: 646 | min: 52 | max: 6210
  Chunking: 6 chunks (by paragraph breaks)

(TV)-Hướng dẫn giao tiếp, ứng xử cho người học.txt
  split  : 6 chunks
  after merge+split: 10 chunks
  too_small (<80): 0
  good (80-800)  : 7
  too_large (>800): 3
  avg: 1043 | min: 90 | max: 4594


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(BASE))

from scripts.chunking import split_text, clean_text
FILE = BASE / "data/text/Quy chế Công tác sinh viên.txt"

with open(FILE, encoding="utf-8") as f:
    raw = f.read()

text   = clean_text(raw)
chunks = split_text(text)

print(f"File : {os.path.basename(FILE)}")
print(f"Gốc  : {len(raw):,} ký tự  →  {len(chunks)} chunks\n")

for i, chunk in enumerate(chunks):
    print(f"─── Chunk {i+1:02d} ({len(chunk)} ký tự) ───")
    print(chunk)
    print()


  Chunking: 35 chunks (by paragraph breaks)
File : Quy chế Công tác sinh viên.txt
Gốc  : 43,932 ký tự  →  35 chunks

─── Chunk 01 (602 ký tự) ───
# QUY CHẾ CÔNG TÁC SINH VIÊN
## CHƯƠNG I: NHỮNG QUY ĐỊNH CHUNG
### Điều 1. Phạm vi và đối tượng áp dụng
1. Quy chế này quy định về công tác sinh viên bao gồm: quyền và nhiệm vụ của sinh viên; học bổng, khen thưởng, hỗ trợ và kỷ luật sinh viên; nội dung công tác sinh viên; hệ thống tổ chức và điều khoản thi hành.
2. Quy chế này là căn cứ pháp lý để điều tiết các hoạt động công tác sinh viên tại Trường Đại học Tôn Đức Thắng.
3. Khoa trong Quy chế này bao gồm các Khoa, các phân hiệu/cơ sở trực thuộc Trường.
4. Quy chế này áp dụng đối với sinh viên được định nghĩa tại Khoản 1 Điều 2 của Quy chế này.

─── Chunk 02 (520 ký tự) ───
### Điều 2. Sinh viên
1. Sinh viên được quy định tại Quy chế này là người học được đào tạo trình độ đại học (chương trình đào tạo tiêu chuẩn, chương trình đào tạo tiên tiến, chương trình liên kết đào tạo quốc tế, chương t

In [ ]:
CHUNKS_PATH  = f'{BASE}/data/chunks/all_chunks.jsonl'
PARENTS_PATH = f'{BASE}/data/chunks/parent_chunks.jsonl'

In [ ]:
# Nếu muốn chạy lại từ đầu (xóa file cũ):
# import os
# for p in [CHUNKS_PATH, PARENTS_PATH]:
#     if os.path.exists(p): os.remove(p)

all_parents, all_chunks = chunker.process_all(
    text_dir=TEXT_DIR,
    txt_files=txt_files,
    chunks_path=CHUNKS_PATH,
    parents_path=PARENTS_PATH,
)

print(f'\nParent chunks : {len(all_parents)}')
print(f'Child chunks  : {len(all_chunks)}')


Processing: (TV)-Hướng dẫn giao tiếp, ứng xử cho người học.txt
  Chunking: 6 chunks (by paragraph breaks)
  Parents: 6 | Children: 10
  ✓ 6 parents | 10 chunks | 3 API calls

Processing: 11.2457-QD cap chung nhan ky su cu nhan uu tu.txt
  Chunking: 5 chunks (by paragraph breaks)
  Parents: 5 | Children: 14
  ✓ 5 parents | 14 chunks | 4 API calls

Processing: 15.2018 - Quy dinh ve hoat dong tap su nghe nghiep.txt
  Chunking: 8 chunks (by paragraph breaks)
  Parents: 8 | Children: 26
  ✓ 8 parents | 26 chunks | 7 API calls

Processing: 17.Thong tin phong ban.txt
  Chunking: 15 chunks (by paragraph breaks)
  Parents: 15 | Children: 34
  ✓ 15 parents | 34 chunks | 9 API calls

Processing: 2021_Quy chế tổ chức và quản lý đào tạo.txt
  Chunking: 38 chunks (by paragraph breaks)
  Parents: 38 | Children: 168
  ✓ 38 parents | 168 chunks | 42 API calls

Processing: 25.2020-Noi quy phong thi.txt
  Chunking: 7 chunks (by paragraph breaks)
  Parents: 7 | Children: 14
  ✓ 7 parents | 14 chunks | 4 

## 6. Statistics

In [ ]:
from collections import Counter
import json

# Reload từ file
with open(CHUNKS_PATH,  'r', encoding='utf-8') as f:
    all_chunks = [json.loads(l) for l in f if l.strip()]
with open(PARENTS_PATH, 'r', encoding='utf-8') as f:
    all_parents = [json.loads(l) for l in f if l.strip()]

if not all_chunks:
    print('⚠️  Không có chunks nào — kiểm tra lỗi ở cell process_all.')
else:
    lengths = [len(c['text']) for c in all_chunks]
    p_lengths = [len(p['text']) for p in all_parents]

    print('=== Chunk Statistics ===')
    print(f'Parent chunks: {len(all_parents)} | avg {sum(p_lengths)//len(p_lengths)} chars')
    print(f'Child chunks : {len(all_chunks)}  | avg {sum(lengths)//len(lengths)} chars')
    print(f'Child min/max: {min(lengths)} / {max(lengths)} chars')
    print()

    labeled = [c for c in all_chunks if c.get('title')]
    print(f'Labeled (title+summary): {len(labeled)}/{len(all_chunks)}')

    if labeled:
        s = labeled[0]
        print(f'\n--- Sample child chunk ---')
        print(f'chunk_id        : {s["chunk_id"]}')
        print(f'parent_chunk_id : {s["parent_chunk_id"]}')
        print(f'title           : {s["title"]}')
        print(f'summary         : {s["summary"]}')
        print(f'text (child)    : {s["text"][:150]}...')

        parent = next((p for p in all_parents if p["parent_chunk_id"] == s["parent_chunk_id"]), None)
        if parent:
            print(f'parent text     : {parent["text"][:150]}...')

    print()
    print('Child chunks per source file:')
    per_file = Counter(c['source_file'] for c in all_chunks)
    for fname, count in sorted(per_file.items(), key=lambda x: -x[1]):
        print(f'  {count:3d}  {fname}')

=== Chunk Statistics ===
Parent chunks: 232 | avg 1582 chars
Child chunks : 850  | avg 474 chars
Child min/max: 46 / 6210 chars

Labeled (title+summary): 848/850

--- Sample child chunk ---
chunk_id        : (TV)-Hướng_dẫn_giao_tiếp,_ứng__chunk_0000
parent_chunk_id : (TV)-Hướng_dẫn_giao_tiếp,_ứng__parent_0000
title           : Hướng dẫn giao tiếp ứng xử cho người học
summary         : Hướng dẫn về giao tiếp, ứng xử cho người học tại Trường Đại học Tôn Đức Thắng, nêu rõ mục đích.
text (child)    : HƯỚNG DẪN
Về giao tiếp, ứng xử cho người học tại Trường Đại học Tôn Đức Thắng
I. Mục đích:...
parent text     : HƯỚNG DẪN
Về giao tiếp, ứng xử cho người học tại Trường Đại học Tôn Đức Thắng
I. Mục đích:
- Gìn giữ truyền thống tốt đẹp của tập thể viên chức, người...

Child chunks per source file:
  174  Quy chế Công tác sinh viên.txt
  168  2021_Quy chế tổ chức và quản lý đào tạo.txt
  168  Hướng dẫn học bổng, khen thưởng, hỗ trợ người học và hỗ trợ khác.txt
   46  QĐ số 22 vv ban hành quy định

---
**Next step:** Run `02_qa_generation.ipynb` để sinh QA pairs từ toàn bộ chunks.  